<a href="https://colab.research.google.com/github/vishisht-gupta/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the target label
df["is_declining_label"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

print(df.shape[0], "pages | declining rate:", round(df["is_declining_label"].mean(), 3))

# -----------------------------
# Hand-written baseline rule
# -----------------------------

# A page is considered stale if it has not been updated in 180+ days
stale = (df["days_since_last_update"] >= 180).astype(int)

# A page is considered visible if it still receives at least 500 impressions
visible = (df["impressions_90d"] >= 500).astype(int)

# Hand-written score
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

# Top 10 pages according to the rule
top10 = df.sort_values("hand_rule_score", ascending=False).head(10)

# Display the important columns
top10[
    [
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "trend_direction",
        "hand_rule_score",
    ]
]

30000 pages | declining rate: 0.542


,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,hand_rule_score
16751,61678,194,19.7,0.15,down,61678
16514,59472,194,24.8,0.13,down,59472
7021,25715,194,22.2,0.23,down,25715
21268,13299,193,10.5,0.49,down,13299
11489,7812,194,39.0,0.01,down,7812
12045,7558,193,17.9,0.20,down,7558
698,4590,194,31.0,0.00,down,4590
5327,4556,194,16.4,0.33,down,4556
26810,4429,194,25.3,0.38,down,4429
20837,1697,193,15.8,0.12,down,1697


In [2]:
# Precision@K evaluation

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values

for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



In [6]:
tree_score = tree.predict_proba(X)[:, 1]

for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)

    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.700
Precision@50:  hand rule 0.680   vs   tree 0.720


In [7]:
# Experiment 1: Compare different tree depths

from sklearn.tree import DecisionTreeClassifier, export_text

for depth in [2, 3, 4]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    tree.fit(X, y)

    score = tree.predict_proba(X)[:, 1]

    print("=" * 60)
    print(f"Tree Depth = {depth}")
    print(f"Precision@50 = {precision_at_k(score, y, 50):.3f}")
    print(export_text(tree, feature_names=features))

Tree Depth = 2
Precision@50 = 0.600
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0

Tree Depth = 3
Precision@50 = 0.720
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|  

In [8]:
# Experiment 2: Replace impressions_90d with engagement_rate

new_features = [
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "avg_position",
    "ctr",
    "word_count"
]

X2 = df[new_features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree2 = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree2.fit(X2, y)

print(export_text(tree2, feature_names=new_features))

score = tree2.predict_proba(X2)[:, 1]

print("Precision@50:", precision_at_k(score, y, 50))

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

Precision@50: 0.7


In [ ]:
Experiment:
I tested different decision tree depths (2, 3, and 4) and also replaced
'impressions_90d' with 'engagement_rate' as a feature.

Observations:
- Depth 2: Precision@50 = 0.60
- Depth 3: Precision@50 = 0.72
- Depth 4: Precision@50 = 0.68

The depth-3 tree achieved the best Precision@50 while remaining reasonably
interpretable. Increasing the depth to 4 made the tree much more complex,
but performance decreased slightly, suggesting that a deeper tree may begin
to overfit the training data.

When I replaced 'impressions_90d' with 'engagement_rate', the tree selected
'avg_position' as the first split, and Precision@50 was about 0.70. This
shows that changing the available features changes the decision rules learned
by the model.